# House building classifier - analysis
Explores the files written by the scripts, it does not train or predict. Run these first, see the README:
- `train.py --oof-output outputs/train_oof.csv`
- `predict.py --output outputs/scotland_pred.csv` (this notebook reads the newest `outputs/*pred*.csv`)
- `validate.py --errors-output outputs/validation_errors.csv`

In [ ]:
from pathlib import Path

import pandas as pd

from config import add_n_homes
from n_homes import N_HOMES_CLASSES, n_homes_class

pd.set_option("display.max_colwidth", None)

In [ ]:
# out-of-fold predictions for the manually checked rows
oof = pd.read_csv("outputs/train_oof.csv", index_col=0)

# all rows passing the prediction filter, from the most recently written predictions csv
pred_files = list(Path("outputs").glob("*pred*.csv"))
if not pred_files:
    raise FileNotFoundError("No predictions csv in outputs/, run predict.py first")
pred_file = max(pred_files, key=lambda p: p.stat().st_mtime)
print(f"Predictions: {pred_file}")
preds = pd.read_csv(pred_file, low_memory=False, dtype={"n_homes_class": "string"})
preds["n_homes_class"] = pd.Categorical(
    preds["n_homes_class"], categories=N_HOMES_CLASSES, ordered=True
)

# rows where the predictions disagree with the validation labels
errors = pd.read_csv("outputs/validation_errors.csv", low_memory=False)

### False positives - examples with strongest +ve prediction
Some of these appear to be labelling errors?

In [ ]:
# show top 10 where most strongly predicted as housing, but not labelled as housing in spreadsheet
oof[oof["ml_class"] == 0].sort_values("pred_prob", ascending=False)[
    ["proposal_text", "is_housebuilding", "pred_prob"]
].head(10)

### False negative - examples with strongest -ve prediction

In [ ]:
# show top 10 where not predicted as housing, but labelled as housing
oof[oof["ml_class"] == 1].sort_values("pred_prob", ascending=True)[
    ["proposal_text", "is_housebuilding", "pred_prob"]
].head(10)

## Stage 2 - estimate number of homes
Output is one of 11 classes: `1`, `2` ... `10`, `more than 10`

Rule based (regex) rather than ML, because:
- the count is nearly always written in the text, e.g. "erection of 22no. houses", "two dwellinghouses", "14 flatted dwellings"
- ~580 labelled positives, mostly 1 home, leaves very few examples for classes 3-10 to learn from, a regex reads the number directly
- the rules are easy to inspect and change

Rules are in `n_homes.py`, in order:
1. `count` - first number followed by a home noun (house, dwelling, flat, unit...), ignoring storey / bedroom / class / condition / reference numbers
2. `plots` - plot ranges, e.g. "plots 22-25 and 31-32" -> 6
3. `unknown` - plural or outline schemes with no number, e.g. "residential development (in principle)", mixed use -> no class, needs review
4. `single` - singular home noun with no number, e.g. "erection of dwellinghouse" -> 1
5. `none` - no home mention at all, no class (often a sign the stage 1 prediction is a false positive)

The estimated number is then mapped to the 11 classes, anything over 10 is `more than 10`

In [ ]:
# check rules against manually labelled number of homes (true house building rows with a count)
labelled = oof[oof["ml_class"] == 1].copy()
labelled["n_homes_true"] = pd.to_numeric(labelled["n_homes"], errors="coerce")
labelled = add_n_homes(labelled[labelled["n_homes_true"] > 0], "proposal_text")
labelled["n_homes_class_true"] = pd.Categorical(
    labelled["n_homes_true"].map(n_homes_class),
    categories=N_HOMES_CLASSES,
    ordered=True,
)

has_est = labelled[labelled["n_homes_class"].notna()]
print(f"Rows with a class: {len(has_est) / len(labelled):.1%} of {len(labelled)}")
print(
    f"Class accuracy where classified: "
    f"{(has_est['n_homes_class'] == has_est['n_homes_class_true']).mean():.1%}"
)
labelled["n_homes_method"].value_counts()

In [ ]:
# confusion matrix of the 11 classes, rows = labelled, cols = rule based ("no class" = unknown / none)
pd.crosstab(
    labelled["n_homes_class_true"],
    labelled["n_homes_class"].cat.add_categories("no class").fillna("no class"),
    rownames=["actual"],
    colnames=["predicted"],
    dropna=False,
)

In [ ]:
# mismatches - several look like labelling errors, others have no number in the text
labelled[labelled["n_homes_class"] != labelled["n_homes_class_true"]][
    ["proposal_text", "n_homes_true", "n_homes", "n_homes_class", "n_homes_method"]
]

## Scotland predictions

In [ ]:
# the stage 1 predicted house building applications
scot_homes = preds[preds["is_housebuilding"] == 1]
scot_homes["n_homes_class"].value_counts(dropna=False).sort_index()

In [ ]:
# largest schemes - note the same big site can appear in several applications (conditions, s42 etc.)
# so summing n_homes will double count
scot_homes.sort_values("n_homes", ascending=False)[
    ["reference", "local_auth", "proposal", "n_homes"]
].head(10)

## Validation - LLM labelled Scotland extract
`inputs/Planning_Applications__Official_PREMIUM_-_Scotland_labelled.csv` is a copy of the Scotland extract with two added columns, labelled by an LLM reading each `proposal`:
- `is_housebuilding` - 1 if the application relates to creating homes (new build, change of use / conversion to dwellings, renewals, house type changes, conditions / AMSC / NMVs naming dwellings, demolish and replace, PAN / EIA screening for housing), following the same conventions as the checked labels in `planning.xlsx`. Holiday lets, short term lets, HMOs, care homes, student accommodation, annexes, residential caravans and s75 obligation changes are 0
- `n_homes` - number of homes created where stated in the text (singular noun = 1), blank where the text gives no number

This file was not used to train the model, but it is not a fully independent test: some rows have the same proposal text as the training data, and the labels are an LLM's reading, so they can differ from the human reviewers on edge cases. Use the `excluding text seen in training` scores from `validate.py` as the headline stage 1 scores. Only rows that pass the filter are scored

In [ ]:
# false positives - flagged by the model but labelled as not house building, highest probability first
errors[errors["error_type"] == "false_positive"][["proposal", "pred_prob"]].head(10)

In [ ]:
# false negatives - house building the model missed, lowest probability first
errors[errors["error_type"] == "false_negative"].sort_values("pred_prob")[
    ["proposal", "pred_prob"]
].head(10)

In [ ]:
# number of homes class mismatches, on rows both flagged and labelled as house building
errors[errors["error_type"] == "n_homes_mismatch"][
    ["proposal", "n_homes_true", "n_homes", "n_homes_method"]
].head(20)